# 🎵 Comprehensive FFT / DFT Implementation from Scratch
## *A Complete Jupyter Notebook — No Built-in FFT Used*

---

> **Author note:** Every single Fourier transform in this notebook is computed
> by our own hand-written `dft`, `idft`, `fft`, and `ifft` functions.
> `numpy.fft`, `scipy.fft`, and all similar library transforms are **never called**.

---

### 📋 Table of Contents

| # | Section | Topics |
|---|---------|--------|
| 1 | [DFT / IDFT](#sec1) | Definition, twiddle matrix, vectorised O(N²) |
| 2 | [FFT / IFFT](#sec2) | Radix-2 Cooley–Tukey, frequency interpolation |
| 3 | [Large-Number Multiplication](#sec3) | Integer × Integer via convolution |
| 4 | [Polynomial Multiplication](#sec4) | Coefficient arrays, O(N log N) |
| 5 | [ODE Solver](#sec5) | Spectral / pseudo-spectral method |
| 6 | [Signal Analysis](#sec6) | PSD, filtering, AM/FM, spectrograms |
| 7 | [Convolution](#sec7) | Circular, linear, overlap-add |
| 8 | [DFT Property Verification](#sec8) | Linearity, Parseval, shift theorem … |
| 9 | [Calculus via FFT](#sec9) | Spectral derivative & integral |
| 10 | [Image Filters](#sec10) | 2-D FFT, low/high/band-pass, Wiener |
| 11 | [Compression](#sec11) | Signal, image & audio compression |
| 12 | [Extra Applications](#sec12) | CZT, cepstrum, pitch, cross-correlation |


## ⚙️ Imports & Setup

We only use **NumPy** (array maths) and **Matplotlib** (plotting).
No FFT library is imported.


In [ ]:
import numpy as np
import matplotlib
import math

matplotlib.use("Agg")  # headless — change to "inline" in Jupyter
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings

warnings.filterwarnings("ignore")

# ── reproducible noise ────────────────────────────────────────────────────────
RNG = np.random.default_rng(42)

print("NumPy version :", np.__version__)
print("All FFT/DFT functions will be our own implementations — no np.fft used!")

NumPy version : 2.0.1
All FFT/DFT functions will be our own implementations — no np.fft used!


---
<a id="sec1"></a>
## 1 · Discrete Fourier Transform (DFT) & Inverse DFT

### 1.1 Theory

The **Discrete Fourier Transform** converts a finite sequence of $N$ equally-spaced
samples in the *time domain* into the *frequency domain*:

$$X[k] = \sum_{n=0}^{N-1} x[n]\, e^{-j2\pi kn/N}, \quad k = 0, 1, \ldots, N-1$$

The **Inverse DFT** recovers the original signal:

$$x[n] = \frac{1}{N} \sum_{k=0}^{N-1} X[k]\, e^{j2\pi kn/N}$$

#### Twiddle factor
The complex exponential $W_N = e^{-j2\pi/N}$ is called the **twiddle factor**.
The DFT can be written compactly as:

$$X[k] = \sum_{n=0}^{N-1} x[n]\, W_N^{kn}$$

#### Vectorised (matrix) form
For all $k$ at once we can form the $N\times N$ **DFT matrix** $\mathbf{W}$:

$$\mathbf{W}_{kn} = e^{-j2\pi kn/N}$$

so that $\mathbf{X} = \mathbf{W}\,\mathbf{x}$ — one matrix–vector multiply replaces
the double loop. This gives **O(N²)** complexity.

### 1.2 Implementation


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# DFT  —  vectorised, O(N²)
# ─────────────────────────────────────────────────────────────────────────────


def dft(x: np.ndarray) -> np.ndarray:
    """
    Vectorised Discrete Fourier Transform.

    Parameters
    ----------
    x : array_like, shape (N,)
        Input signal (real or complex).

    Returns
    -------
    X : ndarray, shape (N,), complex
        DFT coefficients.

    Algorithm
    ---------
    Build the N×N twiddle matrix W where W[k, n] = exp(-j2π·k·n / N),
    then return W @ x  (single BLAS matrix-vector multiply).
    """
    N = len(x)
    n = np.arange(N)
    k = n[:, None]  # column vector  shape (N,1)
    W = np.exp(-2j * np.pi * k * n / N)  # twiddle matrix shape (N,N)
    return W @ x.astype(complex)


def idft(X: np.ndarray) -> np.ndarray:
    """
    Vectorised Inverse DFT.

    The IDFT matrix is the conjugate of the DFT matrix divided by N.
    """
    N = len(X)
    n = np.arange(N)
    k = n[:, None]
    W = np.exp(2j * np.pi * k * n / N)  # conjugate twiddle
    return (W @ X.astype(complex)) / N


# ── quick test ────────────────────────────────────────────────────────────────
x_test = np.array([1, 2, 3, 4], dtype=float)
X_test = dft(x_test)
x_rec = np.real(idft(X_test))

print("Input signal  :", x_test)
print("DFT output    :", np.round(X_test, 4))
print("IDFT round-trip:", np.round(x_rec, 10))
print("Round-trip error:", np.max(np.abs(x_rec - x_test)))

Input signal  : [1. 2. 3. 4.]
DFT output    : [10.+0.j -2.+2.j -2.-0.j -2.-2.j]
IDFT round-trip: [1. 2. 3. 4.]
Round-trip error: 4.440892098500626e-16


### 1.3 Visualisation — DFT of a Multi-tone Signal

In [3]:
fs = 1000.0
N = 512
t = np.linspace(0, N / fs, N, endpoint=False)

# Signal: 50 Hz + 120 Hz + noise
sig = (
    np.sin(2 * np.pi * 50 * t)
    + 0.5 * np.sin(2 * np.pi * 120 * t)
    + 0.1 * RNG.standard_normal(N)
)

# Compute DFT (use first 64 points for O(N²) speed demo)
X_demo = dft(sig[:64])
freqs = np.arange(64) * fs / 64

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(t[:64], sig[:64])
axes[0].set_title("Time-domain signal (first 64 samples)")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Amplitude")

axes[1].stem(
    freqs[:32], np.abs(X_demo[:32]) / 32, linefmt="b-", markerfmt="bo", basefmt=" "
)
axes[1].set_title("DFT Magnitude Spectrum")
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_ylabel("|X[k]| / N")
axes[1].axvline(50, color="r", linestyle="--", alpha=0.6, label="50 Hz")
axes[1].axvline(120, color="g", linestyle="--", alpha=0.6, label="120 Hz")
axes[1].legend()

plt.tight_layout()
plt.savefig("sec1_dft.png", dpi=120)
plt.show()
print("Figure saved → sec1_dft.png")

Figure saved → sec1_dft.png


---
<a id="sec2"></a>
## 2 · Fast Fourier Transform — Radix-2 Cooley–Tukey

### 2.1 Theory

The DFT naively costs **O(N²)** operations.
The **FFT** exploits the periodicity and symmetry of the twiddle factor to
reduce this to **O(N log₂ N)** — a massive speedup (e.g. N=1024 → 1 048 576 vs 10 240 operations).

#### Divide-and-Conquer decomposition

For $N$ a power of 2, split $x[n]$ into **even** and **odd** indexed sub-sequences:

$$X[k] = \underbrace{\sum_{n=0}^{N/2-1} x[2n]\,W_N^{2kn}}_{E[k]}
       + W_N^k\,\underbrace{\sum_{n=0}^{N/2-1} x[2n+1]\,W_N^{2kn}}_{O[k]}$$

Because $W_N^2 = W_{N/2}$, both sub-sums are themselves DFTs of half-length sequences.
This gives the **butterfly equations**:

$$X[k]       = E[k] + W_N^k \cdot O[k]$$
$$X[k + N/2] = E[k] - W_N^k \cdot O[k]$$

Applying this recursively yields the **Cooley–Tukey radix-2 FFT**.

#### Inverse FFT

Using the identity:

$$\text{IFFT}(X) = \frac{1}{N}\,\overline{\text{FFT}(\bar{X})}$$

where $\bar{\cdot}$ denotes complex conjugation, the inverse is free once we have FFT.

#### Frequency-domain interpolation

Zero-padding the FFT spectrum (inserting zeros in the middle) and taking the
IFFT gives a signal sampled at a *higher rate* — equivalent to sinc-interpolation.

### 2.2 Implementation


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Helper — next power of 2
# ─────────────────────────────────────────────────────────────────────────────


from numpy import ndarray


def _next_pow2(n: int) -> int:
    """Smallest power of 2 that is >= n."""
    p = 1
    while p < n:
        p <<= 1
    return p


def interpolate(x: ndarray, new_length: int) -> ndarray:
    N = len(x)
    old_indices = np.arange(N)
    new_indices = np.linspace(0, N - 1, new_length)
    return np.interp(new_indices, old_indices, x)


# ─────────────────────────────────────────────────────────────────────────────
# FFT — Cooley-Tukey Radix-2 (recursive, vectorised butterfly)
# ─────────────────────────────────────────────────────────────────────────────


def fft(x: np.ndarray) -> np.ndarray:
    """
    Radix-2 Cooley–Tukey FFT.

    Input is zero-padded to the next power of 2 if not already.

    Complexity: O(N log N)

    Key idea
    --------
    Recursively split into even/odd sub-problems then combine with
    the butterfly:
        X[k]       = E[k] + W·O[k]      (W = twiddle factor)
        X[k + N/2] = E[k] - W·O[k]
    """
    N = len(x)
    if N == 1:
        return x.astype(complex)

    # Zero-pad to next power-of-2 if needed
    if not math.log2(N).is_integer():
        x = interpolate(x, _next_pow2(N))
        N = len(x)

    # Recursive split on even / odd indices
    even = fft(x[::2])
    odd = fft(x[1::2])

    # Twiddle factors for first half
    half = N // 2
    k = np.arange(half)
    T = np.exp(-2j * np.pi * k / N) * odd

    # Butterfly combination
    return np.concatenate([even + T, even - T])


def ifft(X: np.ndarray) -> np.ndarray:
    """
    Inverse FFT using the conjugate symmetry identity:
        IFFT(X) = conj(FFT(conj(X))) / N
    """
    return np.conj(fft(np.conj(X.astype(complex)))) / len(X)


# ── accuracy check vs our DFT ─────────────────────────────────────────────────
x_chk = RNG.standard_normal(64)
X_dft = dft(x_chk)
X_fft = fft(x_chk)
err = np.max(np.abs(X_dft - X_fft))
print(f"Max |DFT - FFT| error : {err:.2e}   (should be < 1e-10)")

# round-trip
x_rt = np.real(ifft(X_fft))[:64]
print(f"IFFT round-trip error  : {np.max(np.abs(x_rt - x_chk)):.2e}")

Max |DFT - FFT| error : 3.63e-13   (should be < 1e-10)
IFFT round-trip error  : 4.44e-16


### 2.3 Speed Comparison: DFT (O(N²)) vs FFT (O(N log N))

In [ ]:
import time

sizes = [16, 32, 64, 128, 256, 512]
t_dft, t_fft = [], []

for n in sizes:
    xn = RNG.standard_normal(n)

    t0 = time.perf_counter()
    for _ in range(20):
        dft(xn)
    t_dft.append((time.perf_counter() - t0) / 20 * 1000)

    t0 = time.perf_counter()
    for _ in range(20):
        fft(xn)
    t_fft.append((time.perf_counter() - t0) / 20 * 1000)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sizes, t_dft, "o-", label="DFT  O(N²)")
ax.plot(sizes, t_fft, "s-", label="FFT  O(N log N)")
ax.set_xlabel("Signal length N")
ax.set_ylabel("Time per call (ms)")
ax.set_title("DFT vs FFT — execution time")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("sec2_speed.png", dpi=120)
plt.show()

### 2.4 Frequency-domain Interpolation (Upsampling)

Zero-padding the FFT spectrum is equivalent to **sinc (ideal) interpolation** in time.

In [ ]:
def fft_interpolate(x: np.ndarray, M: int) -> np.ndarray:
    """
    Upsample signal x from N to M samples using frequency-domain zero-padding.

    Steps
    -----
    1. Compute FFT of x  → N spectral bins
    2. Insert (M-N) zeros in the middle of the spectrum (high-frequency zero-pad)
    3. Take IFFT → M-sample signal
    4. Scale by M/N to preserve amplitude

    This is mathematically equivalent to ideal sinc interpolation.
    """
    N = len(x)
    X = fft(x)
    half = N // 2
    # Insert zeros between positive and negative frequency bins
    X_pad = np.concatenate([X[:half], np.zeros(M - N, dtype=complex), X[half:]])
    return np.real(ifft(X_pad)) * (M / N)


# Demo
x32 = sig[:32]
x128 = fft_interpolate(x32, 128)
t32 = np.linspace(0, 32 / fs, 32, endpoint=False)
t128 = np.linspace(0, 32 / fs, 128, endpoint=False)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t32, x32, "o", ms=6, label="Original (32 samples)", zorder=3)
ax.plot(t128, x128, "-", lw=2, label="Interpolated (128 samples via FFT)")
ax.set_title("FFT Interpolation — 32 → 128 samples")
ax.set_xlabel("Time (s)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("sec2_interp.png", dpi=120)
plt.show()

---
<a id="sec3"></a>
## 3 · Large-Number Multiplication via FFT

### 3.1 Theory

Multiplying two $d$-digit integers can be viewed as **polynomial multiplication**
of their digit-coefficient polynomials evaluated at base 10.

For integers $a$ and $b$ with digit representations $a_0 + a_1 \cdot 10 + \cdots$
and $b_0 + b_1 \cdot 10 + \cdots$, their product is the polynomial product
$P(x) = A(x) \cdot B(x)$ evaluated at $x = 10$.

Using FFT, this polynomial multiplication costs **O(d log d)** instead of
the schoolbook O(d²).

#### Algorithm
1. Convert both integers to digit arrays (base 10), lowest digit first.  
2. Zero-pad both arrays to length ≥ (d_a + d_b) — a power of 2.  
3. Compute `FFT(A)`, `FFT(B)`.  
4. Pointwise multiply in frequency domain.  
5. `IFFT` → coefficient array (real parts, rounded to integers).  
6. Carry propagation from least-significant digit upward.

### 3.2 Implementation


In [ ]:
def large_number_multiply(a: int, b: int) -> int:
    """
    Multiply two arbitrarily-large integers using FFT convolution.

    Parameters
    ----------
    a, b : int   (any size, including negative)

    Returns
    -------
    product : int
    """
    digits_a = [int(d) for d in str(abs(a))][::-1]  # LSB first
    digits_b = [int(d) for d in str(abs(b))][::-1]

    # Pad to next power-of-2 large enough to hold all output coefficients
    out_len = len(digits_a) + len(digits_b)
    N = _next_pow2(out_len)

    A = fft(np.array(digits_a + [0] * (N - len(digits_a)), dtype=float))
    B = fft(np.array(digits_b + [0] * (N - len(digits_b)), dtype=float))

    # Convolution theorem: multiply pointwise then IFFT
    C = np.real(ifft(A * B))
    coeffs = [int(round(float(c))) for c in C]

    # ── Carry propagation (pure Python ints — avoids C-long overflow) ─────────
    result, carry = 0, 0
    for i, c in enumerate(coeffs):
        val = int(c) + carry
        carry = val // 10
        result += (val % 10) * (10**i)
    result += carry * (10 ** len(coeffs))

    # Handle sign
    sign = -1 if (a < 0) ^ (b < 0) else 1
    return sign * result


# ── Tests ─────────────────────────────────────────────────────────────────────
tests = [
    (123, 456),
    (99999, 88888),
    (123456789, 987654321),
    (10**15, 10**15),
    (-42, 1000),
]

print(f"{'a':>18}  {'b':>18}  {'FFT result':>25}  Correct?")
print("-" * 80)
for a, b in tests:
    r = large_number_multiply(a, b)
    ok = "✓" if r == a * b else "✗"
    print(f"{a:>18}  {b:>18}  {r:>25}  {ok}")

                 a                   b                 FFT result  Correct?
--------------------------------------------------------------------------------
               123                 456                      56088  ✓
             99999               88888                 8888711112  ✓
         123456789           987654321         121932631112635269  ✓
  1000000000000000    1000000000000000  1000000000000000000000000000000  ✓
               -42                1000                     -42000  ✓


---
<a id="sec4"></a>
## 4 · Polynomial Multiplication via FFT

### 4.1 Theory

Given polynomials

$$P(x) = p_0 + p_1 x + p_2 x^2 + \cdots + p_m x^m$$
$$Q(x) = q_0 + q_1 x + q_2 x^2 + \cdots + q_n x^n$$

their product $R = P \cdot Q$ has degree $m + n$ and coefficients

$$r_k = \sum_{j=0}^{k} p_j\, q_{k-j}$$

— which is exactly the **linear convolution** of the coefficient arrays.

The **FFT-based algorithm**:

1. Zero-pad both coefficient arrays to length $N \geq m + n + 1$ (power of 2).  
2. Compute pointwise product of FFTs.  
3. IFFT gives the output coefficients.

Complexity: **O(N log N)** vs schoolbook **O(m·n)**.

### 4.2 Implementation


In [ ]:
def poly_multiply(p: np.ndarray, q: np.ndarray) -> np.ndarray:
    """
    Multiply two polynomials represented as coefficient arrays
    (index = power, so p[0] is the constant term).

    Returns the coefficient array of p·q (length len(p)+len(q)-1).
    """
    n_out = len(p) + len(q) - 1
    N = _next_pow2(n_out)

    P = fft(np.concatenate([p, np.zeros(N - len(p))]).astype(complex))
    Q = fft(np.concatenate([q, np.zeros(N - len(q))]).astype(complex))

    # Convolution theorem
    return np.real(ifft(P * Q))[:n_out]


def poly_str(coeffs, var="x"):
    """Pretty-print a polynomial."""
    terms = []
    for i, c in enumerate(np.round(coeffs).astype(int)):
        if c == 0:
            continue
        if i == 0:
            terms.append(str(c))
        elif i == 1:
            terms.append(f"{c}{var}")
        else:
            terms.append(f"{c}{var}^{i}")
    return " + ".join(terms) if terms else "0"


def poly_evaluate(coeffs, x_vals):
    """Evaluate polynomial at x_vals using Horner's method."""
    result = np.zeros_like(x_vals, dtype=float)
    for c in reversed(coeffs):
        result = result * x_vals + c
    return result


# ── Examples ──────────────────────────────────────────────────────────────────
examples = [
    (np.array([1, 2, 3]), np.array([4, 5]), "1+2x+3x² times 4+5x"),
    (np.array([1, -1]), np.array([1, 1]), "(1-x)(1+x) = 1-x²"),
    (np.array([1, 0, -1]), np.array([1, 1]), "(1-x²)(1+x)"),
    (np.array([1] * 5), np.array([1] * 5), "sum of x^k  ×  sum of x^k"),
]

for p, q, label in examples:
    pq = poly_multiply(p, q)
    print(f"  {label}")
    print(f"    p  = {poly_str(p)}")
    print(f"    q  = {poly_str(q)}")
    print(f"    pq = {poly_str(pq)}")
    print()

  1+2x+3x² times 4+5x
    p  = 1 + 2x + 3x^2
    q  = 4 + 5x
    pq = 4 + 13x + 22x^2 + 15x^3

  (1-x)(1+x) = 1-x²
    p  = 1 + -1x
    q  = 1 + 1x
    pq = 1 + -1x^2

  (1-x²)(1+x)
    p  = 1 + -1x^2
    q  = 1 + 1x
    pq = 1 + 1x + -1x^2 + -1x^3

  sum of x^k  ×  sum of x^k
    p  = 1 + 1x + 1x^2 + 1x^3 + 1x^4
    q  = 1 + 1x + 1x^2 + 1x^3 + 1x^4
    pq = 1 + 2x + 3x^2 + 4x^3 + 5x^4 + 4x^5 + 3x^6 + 2x^7 + 1x^8



### 4.3 Visual Verification

In [ ]:
# p(x) = (x-1)(x-2)(x-3)  → roots at 1,2,3
# q(x) = x + 1
p_roots = np.poly([1, 2, 3])[::-1]  # numpy poly returns high-degree first
q_c = np.array([1, 1], dtype=float)
pq_c = poly_multiply(p_roots, q_c)

x_eval = np.linspace(-0.5, 4, 300)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(x_eval, poly_evaluate(p_roots, x_eval), label="p(x) = (x-1)(x-2)(x-3)")
axes[0].plot(x_eval, poly_evaluate(q_c, x_eval), label="q(x) = x+1")
axes[0].axhline(0, color="k", lw=0.7)
axes[0].set_title("Factor polynomials")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(
    x_eval, poly_evaluate(pq_c, x_eval), color="purple", label="p(x)·q(x) via FFT"
)
axes[1].axhline(0, color="k", lw=0.7)
axes[1].set_title("Product polynomial")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("sec4_poly.png", dpi=120)
plt.show()

---
<a id="sec5"></a>
## 5 · ODE Solver via FFT (Spectral Method)

### 5.1 Theory

The **spectral method** solves differential equations by working in the
Fourier domain where differentiation becomes *multiplication*.

#### Key idea
If $u(x)$ is a periodic function with Fourier transform $\hat{u}(k)$, then:

$$\frac{d^n u}{dx^n} \;\longleftrightarrow\; (jk)^n \hat{u}(k)$$

So a linear ODE with constant coefficients transforms into simple algebra in
Fourier space!

#### Example: first-order ODE with periodic forcing

$$u'(x) + \alpha\,u(x) = f(x), \quad x \in [0, 2\pi), \text{ periodic BC}$$

Taking the DFT of both sides (using $\hat{u'} = jk\hat{u}$):

$$(jk + \alpha)\,\hat{u}(k) = \hat{f}(k)$$

Solve for $\hat{u}$:

$$\hat{u}(k) = \frac{\hat{f}(k)}{jk + \alpha}$$

Then $u = \text{IFFT}(\hat{u})$ — **no time-stepping required!**

### 5.2 Implementation


In [ ]:
def ode_solver_fft(f_rhs, N: int = 512, L: float = 2 * np.pi, alpha: float = 1.0):
    """
    Solve the linear ODE   u'(x) + alpha * u(x) = f(x)
    on the periodic domain [0, L) using the spectral (FFT) method.

    Parameters
    ----------
    f_rhs  : callable  — right-hand side function f(x)
    N      : int       — number of spatial grid points
    L      : float     — domain length
    alpha  : float     — coefficient of u(x)

    Returns
    -------
    x : ndarray, shape (N,)   — spatial grid
    u : ndarray, shape (N,)   — solution

    Algorithm
    ---------
    1. Evaluate f on uniform grid  →  f_vec
    2. FFT(f_vec)  →  F_hat
    3. Frequency vector: k = 2π/L · [0, 1, ..., N/2, -N/2+1, ..., -1]
    4. Divide by spectral operator:  U_hat = F_hat / (i·k + alpha)
    5. u = IFFT(U_hat)
    """
    x = np.linspace(0, L, N, endpoint=False)
    F = fft(f_rhs(x))

    # Wave-numbers (using numpy fftfreq just for frequency AXIS values,
    # not for any actual transform)
    k_idx = np.arange(N)
    k_idx[k_idx > N // 2] -= N  # map to [-N/2, N/2)
    k = k_idx * (2 * np.pi / L)  # physical wave-numbers

    # Spectral operator: (ik + alpha)
    op = 1j * k + alpha
    op[op == 0] = 1e-12  # guard division by zero

    u = np.real(ifft(F / op))
    return x, u


# ── Solve u' + u = sin(3x),  exact: u = (sin3x - 3cos3x)/10  ─────────────────
def rhs_sin3(x):
    return np.sin(3 * x)


x_ode, u_num = ode_solver_fft(rhs_sin3, N=512, alpha=1.0)
u_exact = (np.sin(3 * x_ode) - 3 * np.cos(3 * x_ode)) / 10.0
# The exact solution has an undetermined constant C·e^{-x}; align DC
u_num -= u_num.mean() - u_exact.mean()

print(f"Max error vs exact solution: {np.max(np.abs(u_num - u_exact)):.2e}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_ode, u_exact, "b-", lw=2.5, label="Exact solution")
axes[0].plot(x_ode, u_num, "r--", lw=1.5, label="FFT spectral solver")
axes[0].set_title("ODE  u′ + u = sin(3x)  on [0, 2π)")
axes[0].set_xlabel("x")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].semilogy(x_ode, np.abs(u_num - u_exact) + 1e-16)
axes[1].set_title("Pointwise error |u_FFT − u_exact|")
axes[1].set_xlabel("x")
axes[1].set_ylabel("Error")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("sec5_ode.png", dpi=120)
plt.show()

Max error vs exact solution: 9.89e-16


### 5.3 Spectral Convergence — Exponential Accuracy

In [ ]:
# Show that error decreases exponentially with N (spectral convergence)
Ns = [8, 16, 32, 64, 128, 256, 512]
errors = []
for Ni in Ns:
    xi, ui = ode_solver_fft(rhs_sin3, N=Ni, alpha=1.0)
    ue = (np.sin(3 * xi) - 3 * np.cos(3 * xi)) / 10
    ui -= ui.mean() - ue.mean()
    errors.append(np.max(np.abs(ui - ue)))

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(Ns, errors, "o-", lw=2)
ax.set_xlabel("N (grid points)")
ax.set_ylabel("Max error")
ax.set_title("Spectral convergence: error vs N")
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.savefig("sec5_convergence.png", dpi=120)
plt.show()

---
<a id="sec6"></a>
## 6 · Signal Analysis — Filtering, Modulation, Demodulation

### 6.1 Theory

#### Power Spectral Density (PSD)
The PSD estimates how signal power is distributed across frequencies:

$$S_{xx}(f) = \frac{1}{N f_s} |X(f)|^2$$

For a one-sided PSD (positive frequencies only), multiply by 2 for $f > 0$.

#### Frequency-domain filtering
Filtering in the time domain (convolution with impulse response) equals
**pointwise multiplication** in the frequency domain:

$$Y(f) = H(f) \cdot X(f)$$

An ideal **brick-wall low-pass filter** has $H(f) = 1$ for $|f| \leq f_c$ and $0$ elsewhere.

#### Amplitude Modulation (AM)
$$s(t) = \bigl[1 + m(t)\bigr] \cos(2\pi f_c t)$$

Demodulate by multiplying again by the carrier and low-pass filtering.

#### Frequency Modulation (FM)
$$s(t) = \cos\!\left(2\pi f_c t + 2\pi k_f \int_0^t m(\tau)\,d\tau\right)$$

#### Short-Time Fourier Transform (STFT) / Spectrogram
Windowed FFT applied to overlapping frames reveals how spectrum evolves in time:

$$\text{STFT}(m, k) = \sum_{n} x[n]\,w[n - m H]\,e^{-j2\pi k n/N_w}$$

where $H$ is the hop size and $w$ is the analysis window.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Signal-analysis toolkit
# ─────────────────────────────────────────────────────────────────────────────


def power_spectral_density(x: np.ndarray, fs: float = 1.0):
    """One-sided PSD via periodogram using our FFT."""
    N = len(x)
    X = fft(x)
    psd = (np.abs(X) ** 2) / (N * fs)
    psd[1:-1] *= 2  # fold negative frequencies
    freqs = np.arange(N // 2 + 1) * fs / N
    return freqs, psd[: N // 2 + 1]


def design_lowpass(N: int, cutoff: float, fs: float) -> np.ndarray:
    """Ideal (brick-wall) low-pass filter mask in frequency domain."""
    f = np.concatenate([np.arange(N // 2 + 1), -np.arange(N // 2 - 1, 0, -1)]) * fs / N
    return (np.abs(f) <= cutoff).astype(float)


def design_bandpass(N: int, f_lo: float, f_hi: float, fs: float) -> np.ndarray:
    """Band-pass filter mask."""
    f = np.concatenate([np.arange(N // 2 + 1), -np.arange(N // 2 - 1, 0, -1)]) * fs / N
    return ((np.abs(f) >= f_lo) & (np.abs(f) <= f_hi)).astype(float)


def apply_filter(x: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Apply a frequency-domain filter mask using our FFT."""
    N = len(x)
    X = fft(x)
    N2 = len(X)
    if len(mask) < N2:
        mask = np.concatenate([mask, np.zeros(N2 - len(mask))])
    return np.real(ifft(X * mask[:N2]))[:N]


def am_modulate(message, fc, fs, t):
    """DSB-AM:  s(t) = (1 + m(t)) cos(2π fc t)"""
    return (1 + message) * np.cos(2 * np.pi * fc * t)


def am_demodulate(signal, fc, fs, t, bw):
    """AM demodulation: mix with carrier + low-pass filter."""
    N = len(signal)
    mixed = signal * np.cos(2 * np.pi * fc * t)
    mask = design_lowpass(N, bw, fs)
    return 2 * apply_filter(mixed, mask)


def fm_modulate(message, fc, kf, fs, t):
    """FM:  s(t) = cos(2π fc t + 2π kf ∫m(τ)dτ)"""
    dt = 1.0 / fs
    integral = np.cumsum(message) * dt
    return np.cos(2 * np.pi * fc * t + 2 * np.pi * kf * integral)


def compute_spectrogram(x, fs, window_size=256, hop=64):
    """STFT magnitude using our FFT, Hann window."""
    window = np.hanning(window_size)
    n_frames = (len(x) - window_size) // hop + 1
    spec = np.zeros((window_size // 2, n_frames))
    for i in range(n_frames):
        frame = x[i * hop : i * hop + window_size] * window
        F = fft(frame)
        spec[:, i] = np.abs(F[: window_size // 2])
    times = np.arange(n_frames) * hop / fs
    freqs = np.arange(window_size // 2) * fs / window_size
    return times, freqs, spec


print("Signal analysis toolkit defined.")

Signal analysis toolkit defined.


In [ ]:
# ── Demo: PSD + filtering ─────────────────────────────────────────────────────
f_psd, psd_vals = power_spectral_density(sig, fs)
mask_lp = design_lowpass(N, cutoff=80, fs=fs)
sig_lp = apply_filter(sig, mask_lp)
mask_bp = design_bandpass(N, f_lo=40, f_hi=130, fs=fs)
sig_bp = apply_filter(sig, mask_bp)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

axes[0, 0].semilogy(f_psd, psd_vals)
axes[0, 0].axvline(80, color="r", linestyle="--", label="LP cutoff 80 Hz")
axes[0, 0].set_title("Power Spectral Density")
axes[0, 0].legend()
axes[0, 0].set_xlabel("Frequency (Hz)")

axes[0, 1].plot(t, sig, alpha=0.5, label="Noisy signal")
axes[0, 1].plot(t, sig_lp, lw=2, label="Low-pass 80 Hz")
axes[0, 1].set_title("Low-pass filtering")
axes[0, 1].legend()
axes[0, 1].set_xlabel("Time (s)")

# AM modulation demo
t_am = np.linspace(0, 1, N, endpoint=False)
msg = 0.8 * np.sin(2 * np.pi * 5 * t_am)
am_s = am_modulate(msg, fc=100, fs=fs, t=t_am)
dem_s = am_demodulate(am_s, fc=100, fs=fs, t=t_am, bw=15)
axes[1, 0].plot(t_am[:200], msg[:200], lw=2, label="Message 5 Hz")
axes[1, 0].plot(t_am[:200], am_s[:200], alpha=0.5, label="AM signal")
axes[1, 0].plot(t_am[:200], dem_s[:200], "--", label="Demodulated")
axes[1, 0].set_title("AM Modulation / Demodulation")
axes[1, 0].legend()

# Spectrogram
chirp = np.sin(2 * np.pi * (30 + 70 * t / t[-1]) * t)
times_s, freqs_s, spec = compute_spectrogram(chirp, fs, window_size=64, hop=8)
axes[1, 1].pcolormesh(times_s, freqs_s, np.log1p(spec), shading="auto", cmap="magma")
axes[1, 1].set_title("Spectrogram — chirp (30→100 Hz)")
axes[1, 1].set_xlabel("Time (s)")
axes[1, 1].set_ylabel("Frequency (Hz)")

plt.tight_layout()
plt.savefig("sec6_signal.png", dpi=120)
plt.show()

---
<a id="sec7"></a>
## 7 · Convolution via FFT

### 7.1 Theory

The **Convolution Theorem** is one of the most important results in signal processing:

$$x[n] \circledast h[n] \;\longleftrightarrow\; X[k] \cdot H[k]$$

Time-domain convolution = **frequency-domain multiplication**.

#### Circular vs Linear Convolution

| Property | Circular (N-point) | Linear |
|----------|--------------------|--------|
| Length of output | N | len(x)+len(h)−1 |
| Frequency relation | exact | requires zero-padding |
| Use case | block processing | standard filtering |

For **linear** convolution with FFT, zero-pad both sequences to length ≥ len(x)+len(h)−1.

#### Overlap-Add Method
For very long signals, split into blocks, convolve each block (FFT), then overlap and add.
This is the standard real-time filtering algorithm.


In [ ]:
def circular_convolution(x: np.ndarray, h: np.ndarray) -> np.ndarray:
    """
    N-point circular convolution where N = max(len(x), len(h)).
    Both sequences are zero-padded to length N before FFT.
    """
    N = max(len(x), len(h))
    pad = lambda a: np.concatenate([a, np.zeros(N - len(a))]).astype(complex)
    return np.real(ifft(fft(pad(x)) * fft(pad(h))))


def linear_convolution(x: np.ndarray, h: np.ndarray) -> np.ndarray:
    """
    Linear convolution via zero-padded FFT.
    Output length = len(x) + len(h) - 1.
    """
    n_out = len(x) + len(h) - 1
    N = _next_pow2(n_out)
    pad = lambda a: np.concatenate([a, np.zeros(N - len(a))]).astype(complex)
    return np.real(ifft(fft(pad(x)) * fft(pad(h))))[:n_out]


def overlap_add(x: np.ndarray, h: np.ndarray, block_size: int = 128) -> np.ndarray:
    """
    Overlap-Add convolution — efficient for long x with short h.

    Algorithm
    ---------
    1. Pre-compute FFT of zero-padded h (length N = next_pow2(block+M-1)).
    2. For each block of x:
         a. Zero-pad block to length N
         b. FFT, multiply by H, IFFT
         c. Add result to output (overlapping region from previous block)
    """
    M = len(h)
    N = _next_pow2(block_size + M - 1)
    H = fft(np.concatenate([h, np.zeros(N - M)]).astype(complex))
    out = np.zeros(len(x) + M - 1)
    for start in range(0, len(x), block_size):
        blk = x[start : start + block_size]
        b = np.concatenate([blk, np.zeros(N - len(blk))]).astype(complex)
        y = np.real(ifft(fft(b) * H))[: len(blk) + M - 1]
        out[start : start + len(y)] += y
    return out


# ── Test & comparison ─────────────────────────────────────────────────────────
x_c = np.array([1, 2, 3, 4, 5], dtype=float)
h_c = np.array([1, 0, -1], dtype=float)  # simple differentiator

lin = linear_convolution(x_c, h_c)
circ = circular_convolution(x_c, h_c)
oa = overlap_add(np.tile(x_c, 10), h_c, block_size=8)
ref = linear_convolution(np.tile(x_c, 10), h_c)

print("Linear convolution       :", np.round(lin, 4))
print("Circular convolution     :", np.round(circ, 4))
print("Overlap-add max error    :", np.max(np.abs(oa[: len(ref)] - ref)))

Linear convolution       : [ 1.  2.  2.  2.  2. -4. -5.]
Circular convolution     : [-2.9796 -5.102  -4.6122 -1.5102  1.5918  2.0816  1.9184  1.7551]
Overlap-add max error    : 1.3322676295501878e-15


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].stem(lin, linefmt="b-", markerfmt="bo", basefmt=" ")
axes[0].set_title("Linear convolution [1,2,3,4,5] * [1,0,-1]")

axes[1].stem(circ, linefmt="r-", markerfmt="rs", basefmt=" ")
axes[1].set_title("Circular convolution (N=5)")

axes[2].plot(ref[:60], lw=2, label="Direct")
axes[2].plot(oa[:60], "--", lw=1.5, label="Overlap-Add")
axes[2].set_title("Overlap-Add on long signal")
axes[2].legend()

plt.tight_layout()
plt.savefig("sec7_conv.png", dpi=120)
plt.show()

---
<a id="sec8"></a>
## 8 · DFT Property Verification

### 8.1 Theory — The Six Fundamental Properties

| Property | Time Domain | Frequency Domain |
|----------|-------------|-----------------|
| **Linearity** | $ax[n] + by[n]$ | $aX[k] + bY[k]$ |
| **Time Shift** | $x[n - m]$ | $X[k]\,e^{-j2\pi km/N}$ |
| **Frequency Shift** | $x[n]\,e^{j2\pi mn/N}$ | $X[k - m]$ |
| **Conjugate Symmetry** | $x[n]$ real | $X[k] = X^*[-k]$ |
| **Parseval's Theorem** | $\sum|x|^2$ | $\tfrac{1}{N}\sum|X|^2$ |
| **Convolution** | $x \circledast y$ | $X[k]\cdot Y[k]$ |
| **Time Reversal** | $x[-n\bmod N]$ | $X^*[k]$ (real x) |

Parseval's theorem states that the DFT is an **energy-preserving** (up to scale)
linear transformation:

$$\sum_{n=0}^{N-1} |x[n]|^2 = \frac{1}{N} \sum_{k=0}^{N-1} |X[k]|^2$$


In [ ]:
def verify_dft_properties(N: int = 64, seed: int = 0) -> dict:
    """
    Numerically verify all fundamental DFT properties.

    Returns
    -------
    dict mapping property name → (passed: bool, max_error: float)
    """
    rng = np.random.default_rng(seed)
    x = rng.standard_normal(N)
    y = rng.standard_normal(N)
    a, b = 2.5, -1.3
    m = 7  # shift amount
    X = dft(x)
    Y = dft(y)
    results = {}

    # 1. Linearity
    err = np.max(np.abs(dft(a * x + b * y) - (a * X + b * Y)))
    results["Linearity"] = (err < 1e-10, err)

    # 2. Time Shift → phase ramp in frequency
    k = np.arange(N)
    err = np.max(np.abs(dft(np.roll(x, m)) - X * np.exp(-2j * np.pi * k * m / N)))
    results["Time Shift"] = (err < 1e-10, err)

    # 3. Conjugate Symmetry for real input: X[k] = conj(X[N-k])
    err = np.max(np.abs(X[1:] - np.conj(X[1:][::-1])))
    results["Conjugate Symmetry"] = (err < 1e-10, err)

    # 4. Parseval's Theorem
    err = abs(np.sum(np.abs(x) ** 2) - np.sum(np.abs(X) ** 2) / N)
    results["Parseval"] = (err < 1e-10, err)

    # 5. Convolution ↔ Multiplication
    xy_conv = np.real(idft(X * Y))
    err = np.max(np.abs(dft(xy_conv) - X * Y))
    results["Convolution"] = (err < 1e-10, err)

    # 6. Time Reversal: x[-n mod N] ↔ conj(X[k])  for real x
    x_rev = np.concatenate([[x[0]], x[1:][::-1]])  # circular reversal
    err = np.max(np.abs(dft(x_rev) - np.conj(X)))
    results["Time Reversal"] = (err < 1e-10, err)

    return results


# Run and display
results = verify_dft_properties(N=128)
print(f"\n{'Property':<30}  {'Max Error':>14}  {'Status'}")
print("-" * 55)
for name, (passed, err) in results.items():
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"{name:<30}  {err:>14.2e}  {status}")


Property                             Max Error  Status
-------------------------------------------------------
Linearity                             2.13e-14  ✅ PASS
Time Shift                            7.48e-13  ✅ PASS
Conjugate Symmetry                    7.74e-13  ✅ PASS
Parseval                              2.98e-13  ✅ PASS
Convolution                           1.32e-11  ✅ PASS
Time Reversal                         1.57e-12  ✅ PASS


In [ ]:
# Visual verification — Parseval's theorem across different signal types
signal_types = {
    "Sine": np.sin(2 * np.pi * 10 * t),
    "Square": np.sign(np.sin(2 * np.pi * 5 * t)),
    "Noise": RNG.standard_normal(N),
    "Chirp": np.sin(2 * np.pi * (10 + 50 * t / t[-1]) * t),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (name, s) in zip(axes.ravel(), signal_types.items()):
    S = fft(s)
    lhs = np.sum(np.abs(s) ** 2)
    rhs = np.sum(np.abs(S) ** 2) / N
    err = abs(lhs - rhs) / lhs * 100
    ax.plot(t, s, lw=1.2)
    ax.set_title(
        f"{name}\nParseval: Σ|x|²={lhs:.2f},  Σ|X|²/N={rhs:.2f},  err={err:.2e}%"
    )
    ax.set_xlabel("Time (s)")
    ax.grid(alpha=0.3)

plt.suptitle("Parseval's Theorem Verification", fontsize=14)
plt.tight_layout()
plt.savefig("sec8_parseval.png", dpi=120)
plt.show()

---
<a id="sec9"></a>
## 9 · Spectral Differentiation & Integration

### 9.1 Theory

The most powerful property of the Fourier transform for numerical analysis:

**Differentiation in time = multiplication by $jk$ in frequency:**

$$\widehat{\frac{d^n f}{dx^n}}(k) = \left(\frac{j2\pi k}{L}\right)^n \hat{f}(k)$$

For a signal sampled at $N$ points on $[0, L)$:

$$\hat{f}'(k) = \frac{2\pi j k}{L} \hat{f}(k)$$

where $k$ ranges over $\{0, 1, \ldots, N/2, -N/2+1, \ldots, -1\}$.

**Integration** = division by $jk$ (set DC component to 0):

$$\hat{F}(k) = \frac{\hat{f}(k)}{j 2\pi k / L}, \quad k \neq 0$$

#### Spectral accuracy
Unlike finite-difference methods (2nd-order accuracy), spectral differentiation
achieves **exponential accuracy** for smooth periodic functions — errors decrease
faster than any power of $1/N$.


In [ ]:
def spectral_derivative(x: np.ndarray, fs: float = 1.0, order: int = 1) -> np.ndarray:
    """
    Compute the n-th derivative of a periodic signal using spectral differentiation.

    Parameters
    ----------
    x     : input signal (must be periodic on its domain)
    fs    : sampling frequency in Hz (= N / domain_length)
    order : derivative order (1, 2, 3, ...)

    Algorithm
    ---------
    1. FFT(x)  → X[k]
    2. Multiply by (j · 2π · k · fs / N)^order
    3. Set k=0 coefficient to 0 (removes constant of integration)
    4. IFFT back to time domain
    """
    N = len(x)
    X = fft(x)
    k = np.arange(N)
    k[k > N // 2] -= N  # negative-frequency wrap
    freq_factor = (2j * np.pi * k * fs / N) ** order
    freq_factor[0] = 0  # DC → 0 for derivative
    return np.real(ifft(X * freq_factor))


def spectral_integral(x: np.ndarray, fs: float = 1.0) -> np.ndarray:
    """
    Compute the anti-derivative of a periodic signal (DC set to 0).

    Algorithm
    ---------
    Divide FFT by (j · 2π · k · fs / N), set k=0 coefficient to 0.
    """
    N = len(x)
    X = fft(x)
    k = np.arange(N)
    k[k > N // 2] -= N
    with np.errstate(divide="ignore", invalid="ignore"):
        factor = np.where(k != 0, 1.0 / (2j * np.pi * k * fs / N), 0)
    return np.real(ifft(X * factor))


# ── Demonstrations ────────────────────────────────────────────────────────────
x_func = np.linspace(0, 2 * np.pi, 512, endpoint=False)
fs_func = 512 / (2 * np.pi)

# sin(x) → cos(x)
f1 = np.sin(x_func)
df1_fft = spectral_derivative(f1, fs=fs_func, order=1)
df1_true = np.cos(x_func)

# sin(x) → -sin(x)  (2nd derivative)
d2f1_fft = spectral_derivative(f1, fs=fs_func, order=2)
d2f1_true = -np.sin(x_func)

# ∫cos(x) = sin(x)
f2 = np.cos(x_func)
if2_fft = spectral_integral(f2, fs=fs_func)
if2_fft -= if2_fft.mean() - f1.mean()  # align constant of integration

print(f"d/dx sin(x)  error : {np.max(np.abs(df1_fft  - df1_true)):.2e}")
print(f"d²/dx² sin(x) error: {np.max(np.abs(d2f1_fft - d2f1_true)):.2e}")
print(f"∫cos(x)dx error    : {np.max(np.abs(if2_fft  - f1)):.2e}")

d/dx sin(x)  error : 1.10e-13
d²/dx² sin(x) error: 1.96e-11
∫cos(x)dx error    : 7.42e-16


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

axes[0, 0].plot(x_func, df1_true, "b-", lw=2.5, label="True: cos(x)")
axes[0, 0].plot(x_func, df1_fft, "r--", lw=1.5, label="FFT derivative")
axes[0, 0].set_title("d/dx sin(x)")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(x_func, d2f1_true, "b-", lw=2.5, label="True: −sin(x)")
axes[0, 1].plot(x_func, d2f1_fft, "r--", lw=1.5, label="FFT 2nd deriv")
axes[0, 1].set_title("d²/dx² sin(x)")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

axes[0, 2].plot(x_func, f1, "b-", lw=2.5, label="True: sin(x)")
axes[0, 2].plot(x_func, if2_fft, "r--", lw=1.5, label="FFT ∫cos(x)dx")
axes[0, 2].set_title("∫cos(x)dx")
axes[0, 2].legend()
axes[0, 2].grid(alpha=0.3)

# Multi-frequency function
f3 = np.sin(3 * x_func) + 0.5 * np.cos(7 * x_func)
df3_fft = spectral_derivative(f3, fs=fs_func)
df3_true = 3 * np.cos(3 * x_func) - 3.5 * np.sin(7 * x_func)
axes[1, 0].plot(x_func, df3_true, "b-", lw=2.5, label="True")
axes[1, 0].plot(x_func, df3_fft, "r--", lw=1.5, label="FFT")
axes[1, 0].set_title("d/dx [sin3x + 0.5cos7x]")
axes[1, 0].legend()

# Spectral vs finite-difference accuracy
Ns = [8, 16, 32, 64, 128, 256, 512]
e_sp, e_fd = [], []
for Ni in Ns:
    xi = np.linspace(0, 2 * np.pi, Ni, endpoint=False)
    fi = np.sin(xi)
    di_s = spectral_derivative(fi, fs=Ni / (2 * np.pi))
    # finite difference (2nd order central)
    di_f = np.gradient(fi, xi)
    e_sp.append(np.max(np.abs(di_s - np.cos(xi))))
    e_fd.append(np.max(np.abs(di_f - np.cos(xi))))

axes[1, 1].loglog(Ns, e_sp, "o-", label="Spectral (exp. convergence)")
axes[1, 1].loglog(Ns, e_fd, "s-", label="Finite diff (2nd order)")
axes[1, 1].set_xlabel("N")
axes[1, 1].set_ylabel("Max error")
axes[1, 1].set_title("Spectral vs FD accuracy")
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3, which="both")

# Definite integral via FFT (DC component)
x_int = np.linspace(0, 2 * np.pi, 512, endpoint=False)
val = np.real(fft(np.sin(x_int))[0]) * (2 * np.pi / 512)
axes[1, 2].text(
    0.5,
    0.5,
    f"∫₀²π sin(x) dx\n\nFFT estimate: {val:.8f}\nTrue value: 0.0",
    ha="center",
    va="center",
    fontsize=14,
    transform=axes[1, 2].transAxes,
    bbox=dict(boxstyle="round", facecolor="lightblue", alpha=0.6),
)
axes[1, 2].set_title("Definite integral via FFT")
axes[1, 2].axis("off")

plt.tight_layout()
plt.savefig("sec9_calculus.png", dpi=120)
plt.show()

---
<a id="sec10"></a>
## 10 · Image Analysis & Filters (2-D FFT)

### 10.1 Theory

The **2-D DFT** extends naturally:

$$F[k, l] = \sum_{m=0}^{M-1} \sum_{n=0}^{N-1} f[m,n]\,
             e^{-j2\pi(km/M + ln/N)}$$

This can be computed by applying 1-D FFT along rows, then columns
(separability of the 2-D exponential kernel).

#### Filtering in 2-D
The **2-D convolution theorem** still holds:

$$g * h \;\longleftrightarrow\; G \cdot H$$

#### Common frequency-domain image filters

| Filter | Frequency mask | Effect |
|--------|---------------|--------|
| Low-pass | $H(k,l) = 1$ for $\sqrt{k^2+l^2} \leq r$ | Blurring, noise removal |
| High-pass | $1 - \text{LP}$ | Edge detection, sharpening |
| Band-pass | $\text{LP}_{r_2} - \text{LP}_{r_1}$ | Texture selection |
| Notch | Zero at specific frequency | Remove periodic noise |
| Gaussian | $e^{-(k^2+l^2)/2\sigma^2}$ | Smooth blurring |

#### Wiener Deconvolution
Given a blurred image $g = f * h + n$, the Wiener filter estimates $f$ as:

$$\hat{F}(k,l) = \frac{H^*(k,l)}{|H(k,l)|^2 + K} \cdot G(k,l)$$

where $K$ is the noise-to-signal power ratio.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2-D FFT built from our 1-D FFT
# ─────────────────────────────────────────────────────────────────────────────


def fft2d(img: np.ndarray) -> np.ndarray:
    """
    2-D FFT by applying 1-D FFT along rows, then columns.
    Uses our custom fft() — no np.fft.
    """
    # FFT along each row
    row_transformed = np.apply_along_axis(fft, 1, img.astype(complex))
    # FFT along each column of the row-transformed array
    return np.apply_along_axis(fft, 0, row_transformed)


def ifft2d(IMG: np.ndarray) -> np.ndarray:
    """2-D IFFT — same separability trick."""
    row_inv = np.apply_along_axis(ifft, 1, IMG)
    return np.apply_along_axis(ifft, 0, row_inv)


def image_lowpass(img: np.ndarray, radius: float) -> np.ndarray:
    """Circular low-pass filter via 2-D FFT."""
    H, W = img.shape
    IMG = fft2d(img.astype(float))
    IMG_s = np.fft.fftshift(IMG)  # shift DC to center for masking
    cy, cx = H // 2, W // 2
    Y, X = np.ogrid[:H, :W]
    mask = (Y - cy) ** 2 + (X - cx) ** 2 <= radius**2
    IMG_s *= mask
    return np.real(ifft2d(np.fft.ifftshift(IMG_s)))


def image_highpass(img: np.ndarray, radius: float) -> np.ndarray:
    """High-pass = original − low-pass."""
    return img.astype(float) - image_lowpass(img, radius)


def gaussian_blur_fft(img: np.ndarray, sigma: float) -> np.ndarray:
    """Gaussian blur via 2-D FFT convolution."""
    H, W = img.shape
    Y, X = np.mgrid[:H, :W]
    kernel = np.exp(-((X - W // 2) ** 2 + (Y - H // 2) ** 2) / (2 * sigma**2))
    kernel /= kernel.sum()
    kernel = np.fft.ifftshift(kernel)  # shift DC to corner
    return np.real(ifft2d(fft2d(img.astype(float)) * fft2d(kernel)))


def wiener_deconvolution(
    blurred: np.ndarray, kernel: np.ndarray, K: float = 0.01
) -> np.ndarray:
    """
    Wiener filter to restore a blurred image.

    Estimate:  F_hat = conj(H) * G / (|H|² + K)
    """
    H_img, W_img = blurred.shape
    kh, kw = kernel.shape
    kpad = np.zeros((H_img, W_img))
    kpad[:kh, :kw] = kernel
    G = fft2d(blurred.astype(float))
    H = fft2d(kpad)
    return np.real(ifft2d(np.conj(H) * G / (np.abs(H) ** 2 + K)))


print("2-D FFT image tools defined.")

2-D FFT image tools defined.


In [ ]:
# ── Create synthetic test image ───────────────────────────────────────────────
H, W = 64, 64
Yg, Xg = np.mgrid[:H, :W]
img = (
    128 + 60 * np.sin(2 * np.pi * 4 * Xg / W) + 40 * np.cos(2 * np.pi * 3 * Yg / H)
).astype(float)
img += RNG.standard_normal((H, W)) * 10
img = np.clip(img, 0, 255)

# Apply filters
img_lp = image_lowpass(img, radius=10)
img_hp = image_highpass(img, radius=10)
img_sh = img + 1.5 * img_hp  # unsharp masking
img_gb = gaussian_blur_fft(img, sigma=3)

k_size = 7
kernel = np.zeros((k_size, k_size))
kernel[k_size // 2, :] = 1.0 / k_size  # horizontal motion blur
blurred = gaussian_blur_fft(img, sigma=2)
img_wr = wiener_deconvolution(blurred, kernel, K=0.05)

# 2-D FFT spectrum
IMG_spec = np.abs(np.fft.fftshift(fft2d(img)))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("2-D FFT Image Filters", fontsize=14)
panels = [
    (img, "Original"),
    (np.log1p(IMG_spec), "2-D FFT Spectrum (log)"),
    (img_lp, "Low-pass  r=10"),
    (img_hp, "High-pass r=10"),
    (np.clip(img_sh, 0, 255), "Sharpened"),
    (img_gb, "Gaussian blur σ=3"),
    (blurred, "Blurred (for Wiener)"),
    (np.clip(img_wr, 0, 255), "Wiener deconv"),
]
for ax, (im, title) in zip(axes.ravel(), panels):
    ax.imshow(im, cmap="gray")
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.savefig("sec10_image.png", dpi=120)
plt.show()

---
<a id="sec11"></a>
## 11 · Signal, Image & Audio Compression via FFT

### 11.1 Theory

The key insight behind transform-based compression:

> **Natural signals are sparse in the frequency domain** — most of the energy is
> concentrated in a small fraction of frequency components.

#### FFT Compression Algorithm

1. Compute FFT of the signal.
2. Sort coefficients by **magnitude** (importance).
3. Zero out the smallest $100(1-k)\%$ coefficients.
4. Store only the $k\%$ non-zero coefficients (index + complex value).
5. Reconstruct via IFFT.

#### Quality Metric — SNR (Signal-to-Noise Ratio)

$$\text{SNR} = 10 \log_{10}\left(\frac{\sum|x|^2}{\sum|x - \hat{x}|^2}\right) \text{ dB}$$

Higher SNR = better reconstruction quality.

#### Why does this work?
The DFT concentrates signal energy. For a signal with $K$ dominant frequencies,
keeping only those $K$ coefficients (and zeroing the rest) perfectly reconstructs
the signal. Real signals are *approximately* sparse — so we get good quality
even with aggressive compression.


In [ ]:
def fft_compress_signal(x: np.ndarray, keep_fraction: float) -> tuple:
    """
    FFT-based 1-D signal compression.

    Parameters
    ----------
    x             : input signal
    keep_fraction : fraction of FFT coefficients to keep (0 < kf ≤ 1)

    Returns
    -------
    x_rec  : reconstructed signal
    ratio  : compression ratio = N / n_kept
    snr_db : reconstruction SNR in dB
    """
    X = fft(x)
    N = len(X)
    n_keep = max(1, int(N * keep_fraction))

    # Keep only the n_keep largest-magnitude coefficients
    idx = np.argsort(np.abs(X))[::-1]
    X_comp = np.zeros(N, dtype=complex)
    X_comp[idx[:n_keep]] = X[idx[:n_keep]]

    x_rec = np.real(ifft(X_comp))
    noise = x - x_rec
    snr = 10 * np.log10(np.sum(x**2) / (np.sum(noise**2) + 1e-12))
    return x_rec, N / n_keep, snr


def fft_compress_image(img: np.ndarray, keep_fraction: float) -> tuple:
    """
    FFT-based 2-D image compression.

    Parameters
    ----------
    img           : grayscale image (H×W float array)
    keep_fraction : fraction of 2-D FFT coefficients to keep

    Returns
    -------
    img_rec : reconstructed image (clipped to [0, 255])
    ratio   : compression ratio
    snr_db  : reconstruction SNR in dB
    """
    IMG = fft2d(img.astype(float))
    N_tot = IMG.size
    n_keep = max(1, int(N_tot * keep_fraction))
    thresh = np.sort(np.abs(IMG).ravel())[::-1][n_keep - 1]

    IMG_comp = IMG.copy()
    IMG_comp[np.abs(IMG) < thresh] = 0

    img_rec = np.clip(np.real(ifft2d(IMG_comp)), 0, 255)
    noise = img.astype(float) - img_rec
    snr = 10 * np.log10(np.sum(img.astype(float) ** 2) / (np.sum(noise**2) + 1e-12))
    return img_rec, N_tot / n_keep, snr


print("Compression functions defined.")

Compression functions defined.


In [ ]:
fractions = [0.01, 0.05, 0.10, 0.20, 0.50]
snrs_s, snrs_i = [], []
for kf in fractions:
    _, _, snr_s = fft_compress_signal(sig, kf)
    _, _, snr_i = fft_compress_image(img, kf)
    snrs_s.append(snr_s)
    snrs_i.append(snr_i)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("FFT Compression", fontsize=14)

# Signal compression at different rates
for kf, color in zip([0.05, 0.10, 0.30], ["r", "g", "b"]):
    rec, ratio, snr = fft_compress_signal(sig, kf)
    axes[0, 0].plot(
        t[:100],
        rec[:100],
        "-",
        color=color,
        alpha=0.8,
        label=f"{kf*100:.0f}% kept  SNR={snr:.1f}dB",
    )
axes[0, 0].plot(t[:100], sig[:100], "k--", lw=1, label="Original", alpha=0.5)
axes[0, 0].set_title("Signal compression")
axes[0, 0].legend(fontsize=8)

axes[0, 1].plot([f * 100 for f in fractions], snrs_s, "o-", lw=2)
axes[0, 1].set_xlabel("% coefficients kept")
axes[0, 1].set_ylabel("SNR (dB)")
axes[0, 1].set_title("Signal: SNR vs compression")
axes[0, 1].axhline(20, color="r", linestyle="--", alpha=0.5, label="20 dB threshold")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Sorted spectrum magnitude (shows sparsity)
mag_sorted = np.sort(np.abs(fft(sig)))[::-1]
axes[0, 2].semilogy(mag_sorted)
axes[0, 2].axvline(int(0.10 * N), color="r", linestyle="--", label="10% mark")
axes[0, 2].set_title("Sorted FFT magnitudes (energy concentration)")
axes[0, 2].set_xlabel("Rank")
axes[0, 2].legend()
axes[0, 2].grid(alpha=0.3)

# Image compression
axes[1, 0].imshow(img, cmap="gray")
axes[1, 0].set_title("Original image")
axes[1, 0].axis("off")

rec5, ratio5, snr5 = fft_compress_image(img, 0.05)
axes[1, 1].imshow(rec5, cmap="gray")
axes[1, 1].set_title(f"5% coefficients kept SNR = {snr5:.1f} dB  (ratio {ratio5:.0f}×)")
axes[1, 1].axis("off")

axes[1, 2].plot([f * 100 for f in fractions], snrs_i, "s-", color="darkorange", lw=2)
axes[1, 2].set_xlabel("% coefficients kept")
axes[1, 2].set_ylabel("SNR (dB)")
axes[1, 2].set_title("Image: SNR vs compression")
axes[1, 2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("sec11_compression.png", dpi=120)
plt.show()

---
<a id="sec12"></a>
## 12 · Additional FFT Applications

### 12.1 Applications Overview

| Application | Key Idea |
|-------------|---------|
| **Cross-correlation** | $R_{xy}[m] = \text{IFFT}(\overline{X} \cdot Y)$ — finds time delay |
| **Chirp-Z Transform** | Generalises DFT to arbitrary contours in the z-plane |
| **Cepstrum** | $c[n] = \text{IFFT}(\log|X(f)|)$ — homomorphic analysis |
| **Pitch Detection** | Peak of autocorrelation = fundamental period |
| **Notch Filter** | Zero out specific frequency bin(s) |
| **Sub-pixel Shift** | Phase ramp in frequency = sub-sample time shift |


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cross-correlation and Autocorrelation
# ─────────────────────────────────────────────────────────────────────────────


def cross_correlate(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """
    Cross-correlation R_xy[m] via FFT:
        R_xy = IFFT( conj(FFT(x)) * FFT(y) )

    The peak of R_xy gives the time delay between x and y.
    """
    n_out = len(x) + len(y) - 1
    N = _next_pow2(n_out)
    pad = lambda a: np.concatenate([a, np.zeros(N - len(a))]).astype(complex)
    return np.real(ifft(np.conj(fft(pad(x))) * fft(pad(y))))


def autocorrelate(x: np.ndarray) -> np.ndarray:
    """Autocorrelation = cross-correlation of signal with itself."""
    return cross_correlate(x, x)


# ─────────────────────────────────────────────────────────────────────────────
# Chirp-Z Transform (CZT) — generalised DFT
# ─────────────────────────────────────────────────────────────────────────────


def chirp_z(x: np.ndarray, M: int, W: complex, A: complex) -> np.ndarray:
    """
    Chirp-Z Transform: evaluates Z-transform of x at M points
        z_k = A · W^{-k},  k = 0, 1, ..., M-1

    When A=1, W=exp(-j2π/N), M=N, this equals the standard DFT.
    Allows zoom-in on any frequency range without increasing N.

    Algorithm (Bluestein's method)
    ------------------------------
    Uses the identity k·n = -½[(k-n)² - k² - n²] to convert the
    chirp-like summation into a convolution.
    """
    N = len(x)
    L = _next_pow2(N + M - 1)

    n = np.arange(N)
    yn = x * (A**-n) * (W ** (n * n / 2))

    # Chirp filter
    h = np.zeros(L, dtype=complex)
    km = np.arange(M)
    h[:M] = W ** (-km * km / 2)
    kn2 = np.arange(N - 1, 0, -1)
    h[L - N + 1 :] = W ** (-kn2 * kn2 / 2)

    yn_pad = np.concatenate([yn, np.zeros(L - N, dtype=complex)])
    g = ifft(fft(yn_pad) * fft(h))
    return g[:M] * (W ** (km * km / 2))


# ─────────────────────────────────────────────────────────────────────────────
# Cepstrum
# ─────────────────────────────────────────────────────────────────────────────


def real_cepstrum(x: np.ndarray) -> np.ndarray:
    """
    Real cepstrum: IFFT(log|FFT(x)|)

    Used for pitch detection, echo removal, homomorphic deconvolution.
    Separates source (voice) from filter (vocal tract) in speech.
    """
    X = fft(x)
    return np.real(ifft(np.log(np.abs(X) + 1e-12)))


# ─────────────────────────────────────────────────────────────────────────────
# Pitch Detection via autocorrelation
# ─────────────────────────────────────────────────────────────────────────────


def detect_pitch(signal: np.ndarray, fs: float) -> float:
    """
    Estimate fundamental frequency (pitch) via autocorrelation.
    The first peak after lag=0 corresponds to the fundamental period.
    """
    ac = autocorrelate(signal)
    N = len(signal)
    lag = np.argmax(ac[20 : N // 2]) + 20
    return fs / lag if lag > 0 else 0.0


# ─────────────────────────────────────────────────────────────────────────────
# Sub-pixel shift via FFT phase ramp
# ─────────────────────────────────────────────────────────────────────────────


def fft_shift(x: np.ndarray, shift: float) -> np.ndarray:
    """
    Sub-sample shift using FFT phase ramp.
    Shift by non-integer number of samples with no aliasing.
    """
    N = len(x)
    k = np.arange(N)
    k[k > N // 2] -= N
    return np.real(ifft(fft(x) * np.exp(-2j * np.pi * k * shift / N)))


# ─────────────────────────────────────────────────────────────────────────────
# Notch filter
# ─────────────────────────────────────────────────────────────────────────────


def notch_filter(
    x: np.ndarray, notch_freq: float, bandwidth: float, fs: float
) -> np.ndarray:
    """Zero out a narrow band around notch_freq."""
    N_orig = len(x)
    X = fft(x)
    N_fft = len(X)
    k = np.arange(N_fft)
    k[k > N_fft // 2] -= N_fft
    freqs = k * fs / N_fft
    mask = (np.abs(np.abs(freqs) - notch_freq) >= bandwidth / 2).astype(float)
    return np.real(ifft(X * mask))[:N_orig]


print("Extra application functions defined.")

Extra application functions defined.


In [ ]:
# ── Demonstrations ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("Section 12 — Additional FFT Applications", fontsize=13)

# 1. Cross-correlation for lag detection
ref = np.zeros(N)
ref[: len(sig[:20])] = sig[:20]
delay = np.roll(ref, 37)
cc = cross_correlate(ref, delay)
axes[0, 0].plot(cc[:120])
axes[0, 0].axvline(37, color="r", linestyle="--", label="True lag=37")
axes[0, 0].set_title("Cross-correlation (lag detection)")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# 2. Autocorrelation
ac = autocorrelate(sig)
axes[0, 1].plot(ac[: N // 4])
axes[0, 1].set_title("Autocorrelation of test signal")
axes[0, 1].grid(alpha=0.3)

# 3. CZT — zoom on specific frequency range
x_czt = np.exp(-0.05 * t[:128]) * np.sin(2 * np.pi * 50 * t[:128])
czt = chirp_z(x_czt, 256, np.exp(-2j * np.pi / 256), 1.0)
axes[0, 2].plot(np.abs(czt))
axes[0, 2].set_title("Chirp-Z Transform")
axes[0, 2].grid(alpha=0.3)

# 4. Notch filter
t2 = np.linspace(0, 1, 1000, endpoint=False)
noisy = (
    np.sin(2 * np.pi * 50 * t2)
    + 0.8 * np.sin(2 * np.pi * 60 * t2)
    + 0.2 * RNG.standard_normal(1000)
)
clean = notch_filter(noisy, notch_freq=60, bandwidth=5, fs=1000)
axes[1, 0].plot(t2[:200], noisy[:200], alpha=0.6, label="With 60 Hz hum")
axes[1, 0].plot(t2[:200], clean[:200], lw=2, label="Notch filtered")
axes[1, 0].set_title("60 Hz Notch Filter")
axes[1, 0].legend()

# 5. Cepstrum
cep = real_cepstrum(sig)
axes[1, 1].plot(cep[: N // 4])
axes[1, 1].set_title("Real Cepstrum")
axes[1, 1].set_xlabel("Quefrency (samples)")
axes[1, 1].grid(alpha=0.3)

# 6. Pitch detection
f0_true = 220.0
fs_p = 8000.0
t_p = np.arange(2048) / fs_p
voice = (
    np.sin(2 * np.pi * f0_true * t_p)
    + 0.5 * np.sin(2 * np.pi * 2 * f0_true * t_p)
    + 0.2 * np.sin(2 * np.pi * 3 * f0_true * t_p)
)
f0_est = detect_pitch(voice, fs_p)
axes[1, 2].plot(t_p[: int(2 / f0_true * fs_p)], voice[: int(2 / f0_true * fs_p)])
axes[1, 2].set_title(f"Pitch: true={f0_true} Hz  estimated≈{f0_est:.1f} Hz")
axes[1, 2].set_xlabel("Time (s)")
axes[1, 2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("sec12_extra.png", dpi=120)
plt.show()

---
## 🏁 Summary & Key Takeaways

### What we built from scratch

| Function | Purpose | Complexity |
|----------|---------|------------|
| `dft(x)` | Vectorised DFT via twiddle matrix | O(N²) |
| `idft(X)` | Vectorised IDFT | O(N²) |
| `fft(x)` | Radix-2 Cooley–Tukey FFT | O(N log N) |
| `ifft(X)` | Inverse via conjugation identity | O(N log N) |
| `fft2d(img)` | 2-D FFT (separable rows × cols) | O(HW log HW) |
| `ifft2d(IMG)` | 2-D IFFT | O(HW log HW) |

### Core concepts

1. **DFT** converts a time-domain signal to its frequency-domain representation.  
   The $N\times N$ twiddle matrix $\mathbf{W}$ lets us compute it as a single matrix-vector product.

2. **FFT** achieves O(N log N) by recursively splitting into even/odd sub-problems
   (Cooley–Tukey radix-2 algorithm). The key insight is the *butterfly operation*.

3. **Convolution theorem**: time-domain convolution = frequency-domain multiplication.
   This underpins filtering, polynomial multiplication, large-number arithmetic, and ODE solving.

4. **Spectral calculus**: differentiation/integration become multiplication/division in the
   Fourier domain, enabling spectrally-accurate numerical calculus.

5. **Sparsity**: natural signals are sparse in the frequency domain, which is why FFT-based
   compression (keep the largest coefficients, discard the rest) works so well.

### The Cooley–Tukey butterfly 🦋

```
E[k]         E[k] + W·O[k]   = X[k]
         ↗↘
O[k]         E[k] - W·O[k]   = X[k+N/2]
```

Where $W = e^{-j2\pi k/N}$ is the twiddle factor.

---
*All transforms in this notebook were computed using our own `fft()` / `dft()` functions.  
`numpy.fft` was **never called** for any actual Fourier transform.*
